In [200]:
# Libraries
import numpy as np
import pandas as pd
from scipy import stats

In [201]:
# Master tables.
masterTableHuman = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/humanParalogy.txt", sep="\t")
masterTableMouse = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/mouseParalogy.txt", sep="\t")

# Add Expression Profile Information

In [202]:
gtexEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/gtexExpressionProfile.parquet")
emtabEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/emtabExpressionProfile.parquet")

# Merge the paralogue table with the expression profile data. Have to do this twice for the two genes.
masterTableHuman = masterTableHuman.merge(gtexEP.add_prefix("Gene 1 "), on="Gene stable ID", how="left")
masterTableHuman = masterTableHuman.merge(gtexEP.add_prefix("Gene 2 "), left_on="Human paralogue gene stable ID", right_on="Gene stable ID", how="left")

# Merge the paralogue table with the expression profile data. Have to do this twice for the two genes. 
masterTableMouse = masterTableMouse.merge(emtabEP.add_prefix("Gene 1 "), on="Gene stable ID", how="left")
masterTableMouse = masterTableMouse.merge(emtabEP.add_prefix("Gene 2 "), left_on="Mouse paralogue gene stable ID", right_on="Gene stable ID", how="left")

# Calculate Distances

In [203]:
def calcTEC(geneOne, geneTwo):
    # Turns the expression profiles into a binary vector that tells us whether a gene is expressed (TPM > 1) or not in a tissue. Turn the dataframe into a series.
    humanOrthoBinary = (geneOne[:-1] > 1)
    mouseOrthoBinary = (geneTwo[:-1] > 1)

    # Finds the number of tissues that are found in one species and not in the other.
    humanOnlyTissueNum = (humanOrthoBinary & ~mouseOrthoBinary).sum()
    mouseOnlyTissueNum = (mouseOrthoBinary & ~humanOrthoBinary).sum()

    # Using the binary vector, we can calculate the total number of tissues the gene is expressed in.
    humanTotalTissue = humanOrthoBinary.sum()
    mouseTotalTissue = mouseOrthoBinary.sum()

    # If any gene is not expressed in any tissue, the TEC formula will output an error. We handle this case by outputting NaN.
    if humanTotalTissue == 0 or mouseTotalTissue == 0:
        return np.nan
    else:
        return ((humanOnlyTissueNum / humanTotalTissue) + (mouseOnlyTissueNum / mouseTotalTissue)) / 2

In [204]:
# Grabs the parental and daughter copy expression profiles.
geneOneEPHuman = masterTableHuman.filter(like="Gene 1")
geneTwoEPHuman = masterTableHuman.filter(like="Gene 2")

# Calculates Euclidean distance.
masterTableHuman["EuclidDist"] = np.linalg.norm(geneOneEPHuman.select_dtypes(include="number").to_numpy() - geneTwoEPHuman.select_dtypes(include="number").to_numpy(), axis=1)

# Calculates Euclidean distance with Euclidean normalization applied. First calculates the Euclidean norm of each row (axis=1), then divides each "index" (row) by its respective Euclidean norm. 
masterTableHuman["EuclidDistNorm"] = np.linalg.norm(geneOneEPHuman.select_dtypes(include="number").div(np.linalg.norm(geneOneEPHuman.select_dtypes(include="number"), axis=1), axis=0).to_numpy() - geneTwoEPHuman.select_dtypes(include="number").div(np.linalg.norm(geneTwoEPHuman.select_dtypes(include="number"), axis=1), axis=0).to_numpy(), axis=1)

# Calculates Euclidean distance with log2 transformation applied.
masterTableHuman["EuclidDistLog"] = np.linalg.norm(np.log2(geneOneEPHuman.select_dtypes(include="number") + 1).to_numpy() - np.log2(geneTwoEPHuman.select_dtypes(include="number") + 1).to_numpy(), axis=1)                                                                                                                                                                                                                                                                                                                                                                                                                                                                

# Calculates Pearson distance. The "corrwith" function requires dataframes to have the same column name. That's why I had to rename geneTwoEPHuman's columns to geneOneEPHuman'
masterTableHuman["PearDist"] = 1 - geneOneEPHuman.select_dtypes(include="number").corrwith(geneTwoEPHuman.select_dtypes(include="number").rename(columns=dict(zip(geneTwoEPHuman.columns, geneOneEPHuman.columns))), axis=1, method="pearson").to_numpy()

# Iterates through each row in both dataframes and calculates their TEC score.
masterTableHuman["TEC"] = [calcTEC(parentCopy, geneTwoCopy) for parentCopy, geneTwoCopy in zip(geneOneEPHuman.select_dtypes(include="number").to_numpy(), geneTwoEPHuman.select_dtypes(include="number").to_numpy())]

In [205]:
# Grabs the geneOne and geneTwo copy expression profiles.
geneOneEPMouse = masterTableMouse.filter(like="Gene 1")
geneTwoEPMouse = masterTableMouse.filter(like="Gene 2")

# Calculates Euclidean distance.
masterTableMouse["EuclidDist"] = np.linalg.norm(geneOneEPMouse.select_dtypes(include="number").to_numpy() - geneTwoEPMouse.select_dtypes(include="number").to_numpy(), axis=1)

# Calculates Euclidean distance with Euclidean normalization applied. First calculates the Euclidean norm of each row (axis=1), then divides each "index" (row) by its respective Euclidean norm. 
masterTableMouse["EuclidDistNorm"] = np.linalg.norm(geneOneEPMouse.select_dtypes(include="number").div(np.linalg.norm(geneOneEPMouse.select_dtypes(include="number"), axis=1), axis=0).to_numpy() - geneTwoEPMouse.select_dtypes(include="number").div(np.linalg.norm(geneTwoEPMouse.select_dtypes(include="number"), axis=1), axis=0).to_numpy(), axis=1)

# Calculates Euclidean distance with log2 transformation applied.
masterTableMouse["EuclidDistLog"] = np.linalg.norm(np.log2(geneOneEPMouse.select_dtypes(include="number") + 1).to_numpy() - np.log2(geneTwoEPMouse.select_dtypes(include="number") + 1).to_numpy(), axis=1)                                                                                                                                                                                                                                                                                                                                                                                                                                                                

# Calculates Pearson distance. The "corrwith" function requires dataframes to have the same column name. That's why I had to rename geneTwoEPMouse's columns to geneOneEPMouse'
masterTableMouse["PearDist"] = 1 - geneOneEPMouse.select_dtypes(include="number").corrwith(geneTwoEPMouse.select_dtypes(include="number").rename(columns=dict(zip(geneTwoEPMouse.columns, geneOneEPMouse.columns))), axis=1, method="pearson").to_numpy()

# Iterates through each row in both dataframes and calculates their TEC score.
masterTableMouse["TEC"] = [calcTEC(parentCopy, geneTwoCopy) for parentCopy, geneTwoCopy in zip(geneOneEPMouse.select_dtypes(include="number").to_numpy(), geneTwoEPMouse.select_dtypes(include="number").to_numpy())]

# Number of Duplicates

In [206]:
humanPairs = masterTableHuman.iloc[:, 0:2]
mousePairs = masterTableMouse.iloc[:, 0:2]

In [207]:
masterTableHuman = masterTableHuman.merge(humanPairs.groupby("Gene stable ID").count(), left_on="Gene stable ID", right_index=True, how="left").rename(columns={"Human paralogue gene stable ID_y": "Number of Duplicates"})
masterTableMouse = masterTableMouse.merge(mousePairs.groupby("Gene stable ID").count(), left_on="Gene stable ID", right_index=True, how="left").rename(columns={"Mouse paralogue gene stable ID_y": "Number of Duplicates"})

# Exons

In [208]:
humanExons = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/humanExons.txt", sep="\t")
mouseExons = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/mouseExons.txt", sep="\t")

humanExonsGrouped = humanExons.groupby("Gene stable ID")["Exon stable ID"].agg(lambda x: ", ".join(x.unique()))
mouseExonsGrouped = mouseExons.groupby("Gene stable ID")["Exon stable ID"].agg(lambda x: ", ".join(x.unique()))

In [209]:
masterTableHuman = masterTableHuman.merge(humanExonsGrouped, on="Gene stable ID", how="left")
masterTableMouse = masterTableMouse.merge(mouseExonsGrouped, on="Gene stable ID", how="left")

In [210]:
masterTableHuman

,Gene stable ID,Human paralogue gene stable ID_x,Human paralogue homology type,Paralogue last common ancestor with Human,Gene 1 Brain,Gene 1 Colon,Gene 1 Esophagus,Gene 1 Heart,Gene 1 Kidney,Gene 1 Liver,...,Gene 2 Pancreas,Gene 2 Stomach,Gene 2 Gene type,EuclidDist,EuclidDistNorm,EuclidDistLog,PearDist,TEC,Number of Duplicates,Exon stable ID
0,ENSG00000210049,NaN,NaN,NaN,17.516844,1.400129,0.999184,3.366039,4.574239,0.597851,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,ENSE00001544501
1,ENSG00000211459,NaN,NaN,NaN,23502.966797,3544.858643,3435.218750,7646.939941,10045.823242,3707.004150,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,ENSE00001544499
2,ENSG00000210077,NaN,NaN,NaN,18.865494,1.571674,1.380163,3.266439,3.701754,0.723936,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,ENSE00001544498
3,ENSG00000210082,NaN,NaN,NaN,72589.617188,18863.207031,16807.601562,39093.738281,51649.222656,20325.992188,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,ENSE00001544497
4,ENSG00000209082,NaN,NaN,NaN,25.209278,6.738225,4.899840,15.493462,11.392898,4.587103,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,ENSE00002006242
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3625954,ENSG00000157873,ENSG00000243509,other_paralog,Chordata,12.655311,120.186638,69.148285,20.617067,101.015106,54.591118,...,1.235028,4.021246,protein_coding,189.903580,0.500351,10.446088,0.463805,0.142857,21,"ENSE00001759361, ENSE00001576845, ENSE00003529..."
3625955,ENSG00000157873,ENSG00000026103,other_paralog,Chordata,12.655311,120.186638,69.148285,20.617067,101.015106,54.591118,...,0.426707,1.094653,protein_coding,196.172974,0.323165,12.601530,0.198451,0.214286,21,"ENSE00001759361, ENSE00001576845, ENSE00003529..."
3625956,ENSG00000157873,ENSG00000120949,other_paralog,Chordata,12.655311,120.186638,69.148285,20.617067,101.015106,54.591118,...,0.078418,1.143263,protein_coding,198.702774,0.713202,14.173954,0.937936,0.428571,21,"ENSE00001759361, ENSE00001576845, ENSE00003529..."
3625957,ENSG00000132676,NaN,NaN,NaN,23.821829,46.711121,42.119331,28.719112,30.156294,29.079199,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,"ENSE00001936122, ENSE00001845528, ENSE00001811..."


In [211]:
masterTableHuman["Number of Exons"] = masterTableHuman["Exon stable ID"].str.split(", ").str.len()
masterTableMouse["Number of Exons"] = masterTableMouse["Exon stable ID"].str.split(", ").str.len()

# Identifying In/Out-Paralogs

In [212]:
masterTableMouse["Paralogue last common ancestor with Mouse"].unique()

<ArrowStringArray>
[                                      nan,
                               'Bilateria',
 'Mus musculus reference (CL57BL6) strain',
                                     'Mus',
                                 'Murinae',
                                'Muroidea',
                                'Rodentia',
                                'Eutheria',
                           'Gnathostomata',
                              'Vertebrata',
                                'Mammalia',
                           'Boreoeutheria',
                            'Euteleostomi',
                                'Chordata',
                                 'Amniota',
                            'Opisthokonta',
                                  'Theria',
                           'Sarcopterygii',
                               'Tetrapoda',
                               'Myomorpha',
                        'Euarchontoglires',
                                  'Glires']
Length: 22, d

In [213]:
humanSpecific = ["Primates", "Haplorrhini", "Simiiformes", "Catarrhini", "Hominidae", "Homininae", "Homo sapiens"]
mouseSpecific = ["Glires", "Rodentia", "Myomorpha", "Muroidea", "Murinae", "Mus", "Mus musculus reference (CL57BL6) strain"]

humanParalogyConditions = [
    masterTableHuman["Paralogue last common ancestor with Human"].isin(humanSpecific),
    (~masterTableHuman["Paralogue last common ancestor with Human"].isin(humanSpecific)) & (~masterTableHuman["Paralogue last common ancestor with Human"].isna()),
]
mouseParalogyConditions = [
    masterTableMouse["Paralogue last common ancestor with Mouse"].isin(mouseSpecific),
    (~masterTableMouse["Paralogue last common ancestor with Mouse"].isin(mouseSpecific)) & (~masterTableMouse["Paralogue last common ancestor with Mouse"].isna()),
]

paralogyChoices = [
    "in-paralog",
    "out-paralog"
]

masterTableHuman["Paralogy Type"] = np.select(humanParalogyConditions, paralogyChoices, default="NA")
masterTableMouse["Paralogy Type"] = np.select(mouseParalogyConditions, paralogyChoices, default="NA")

masterTableHuman = masterTableHuman.rename(columns={"Human paralogue gene stable ID_x": "Human paralogue gene stable ID"})
masterTableMouse = masterTableMouse.rename(columns={"Mouse paralogue gene stable ID_x": "Mouse paralogue gene stable ID"})

In [214]:
display(masterTableHuman)

,Gene stable ID,Human paralogue gene stable ID,Human paralogue homology type,Paralogue last common ancestor with Human,Gene 1 Brain,Gene 1 Colon,Gene 1 Esophagus,Gene 1 Heart,Gene 1 Kidney,Gene 1 Liver,...,Gene 2 Gene type,EuclidDist,EuclidDistNorm,EuclidDistLog,PearDist,TEC,Number of Duplicates,Exon stable ID,Number of Exons,Paralogy Type
0,ENSG00000210049,NaN,NaN,NaN,17.516844,1.400129,0.999184,3.366039,4.574239,0.597851,...,NaN,NaN,NaN,NaN,NaN,NaN,0,ENSE00001544501,1,NA
1,ENSG00000211459,NaN,NaN,NaN,23502.966797,3544.858643,3435.218750,7646.939941,10045.823242,3707.004150,...,NaN,NaN,NaN,NaN,NaN,NaN,0,ENSE00001544499,1,NA
2,ENSG00000210077,NaN,NaN,NaN,18.865494,1.571674,1.380163,3.266439,3.701754,0.723936,...,NaN,NaN,NaN,NaN,NaN,NaN,0,ENSE00001544498,1,NA
3,ENSG00000210082,NaN,NaN,NaN,72589.617188,18863.207031,16807.601562,39093.738281,51649.222656,20325.992188,...,NaN,NaN,NaN,NaN,NaN,NaN,0,ENSE00001544497,1,NA
4,ENSG00000209082,NaN,NaN,NaN,25.209278,6.738225,4.899840,15.493462,11.392898,4.587103,...,NaN,NaN,NaN,NaN,NaN,NaN,0,ENSE00002006242,1,NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3625954,ENSG00000157873,ENSG00000243509,other_paralog,Chordata,12.655311,120.186638,69.148285,20.617067,101.015106,54.591118,...,protein_coding,189.903580,0.500351,10.446088,0.463805,0.142857,21,"ENSE00001759361, ENSE00001576845, ENSE00003529...",79,out-paralog
3625955,ENSG00000157873,ENSG00000026103,other_paralog,Chordata,12.655311,120.186638,69.148285,20.617067,101.015106,54.591118,...,protein_coding,196.172974,0.323165,12.601530,0.198451,0.214286,21,"ENSE00001759361, ENSE00001576845, ENSE00003529...",79,out-paralog
3625956,ENSG00000157873,ENSG00000120949,other_paralog,Chordata,12.655311,120.186638,69.148285,20.617067,101.015106,54.591118,...,protein_coding,198.702774,0.713202,14.173954,0.937936,0.428571,21,"ENSE00001759361, ENSE00001576845, ENSE00003529...",79,out-paralog
3625957,ENSG00000132676,NaN,NaN,NaN,23.821829,46.711121,42.119331,28.719112,30.156294,29.079199,...,NaN,NaN,NaN,NaN,NaN,NaN,0,"ENSE00001936122, ENSE00001845528, ENSE00001811...",131,NA


# dN/dS and Proteins

In [215]:
humanParalogdNdS = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/OmegaDnDs/paralogsHumanOmegaDnDs.txt", sep="\t", header=None)
mouseParalogdNdS = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/OmegaDnDs/paralogsMouseOmegaDnDs.txt", sep="\t", header=None)     
humanParalogProtID = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/proteinIdentity/paralogsHumanProtId.txt", sep="\t", header=None) 
mouseParalogProtID = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/proteinIdentity/paralogsMouseProtId.txt", sep="\t", header=None) 

humanParalogdNdS[["Gene 1", "Gene 2"]] = humanParalogdNdS.pop(0).str.split("__", expand=True)
humanParalogdNdS = humanParalogdNdS.iloc[:, [3, 4, 0, 1, 2]]
mouseParalogdNdS[["Gene 1", "Gene 2"]] = mouseParalogdNdS.pop(0).str.split("__", expand=True)
mouseParalogdNdS = mouseParalogdNdS.iloc[:, [3, 4, 0, 1, 2]]

humanParalogProtID[["Gene 1", "Gene 2"]] = humanParalogProtID.pop(0).str.split("__", expand=True)
humanParalogProtID = humanParalogProtID.iloc[:, [1, 2, 0]]
mouseParalogProtID[["Gene 1", "Gene 2"]] = mouseParalogProtID.pop(0).str.split("__", expand=True)
mouseParalogProtID = mouseParalogProtID.iloc[:, [1, 2, 0]]

display(humanParalogdNdS)
display(mouseParalogProtID)

,Gene 1,Gene 2,1,2,3
0,ENSG00000107807,ENSG00000105997,0.00494,0.7813,158.0690
1,ENSG00000186509,ENSG00000280314,0.01760,0.4857,27.6009
2,ENSG00000188993,ENSG00000132000,0.00726,0.9528,131.2690
3,ENSG00000149133,ENSG00000172457,0.03630,0.4088,11.2615
4,ENSG00000109654,ENSG00000158022,0.00654,0.7137,109.1384
...,...,...,...,...,...
132563,ENSG00000139323,ENSG00000163811,0.00723,0.9656,133.4958
132564,ENSG00000107807,ENSG00000180806,0.03473,0.7469,21.5022
132565,ENSG00000279116,ENSG00000231192,0.02259,0.5860,25.9406
132566,ENSG00000088205,ENSG00000111364,0.06526,0.5791,8.8739


,Gene 1,Gene 2,1
0,ENSMUSG00000000001,ENSMUSG00000034781,52.285714
1,ENSMUSG00000000001,ENSMUSG00000034792,46.438746
2,ENSMUSG00000000003,ENSMUSG00000015090,27.814570
3,ENSMUSG00000000003,ENSMUSG00000045684,28.275862
4,ENSMUSG00000000003,ENSMUSG00000047356,26.174497
...,...,...,...
348509,ENSMUSG00002076083,ENSMUSG00000043929,26.094571
348510,ENSMUSG00002076083,ENSMUSG00000070923,34.347826
348511,ENSMUSG00002076083,ENSMUSG00000073700,35.335689
348512,ENSMUSG00002076083,ENSMUSG00000074001,33.264463


In [ ]:
masterTableHuman = masterTableHuman.merge(humanParalogdNdS, left_on=["Gene stable ID", "Human paralogue gene stable ID"], right_on=["Gene 1", "Gene 2"], how="left").drop(columns=["Gene 1", "Gene 2"]).rename(columns={1: "dN/dS", 2:"dN", 3:"dS"})
masterTableHuman = masterTableHuman.merge(humanParalogdNdS, left_on=["Gene stable ID", "Human paralogue gene stable ID"], right_on=["Gene 2", "Gene 1"], how="left").drop(columns=["Gene 1", "Gene 2"]).rename(columns={1: "dN/dS 1", 2:"dN 1", 3:"dS 1"})
masterTableHuman["dN/dS"] = masterTableHuman["dN/dS"].fillna(masterTableHuman["dN/dS 1"])
masterTableHuman["dN"] = masterTableHuman["dN"].fillna(masterTableHuman["dN 1"])
masterTableHuman["dS"] = masterTableHuman["dS"].fillna(masterTableHuman["dS 1"])
masterTableHuman.drop(columns=["dN/dS 1", "dN 1", "dS 1"], inplace=True)

masterTableMouse = masterTableMouse.merge(mouseParalogdNdS, left_on=["Gene stable ID", "Mouse paralogue gene stable ID"], right_on=["Gene 1", "Gene 2"], how="left").drop(columns=["Gene 1", "Gene 2"]).rename(columns={1: "dN/dS", 2:"dN", 3:"dS"})
masterTableMouse = masterTableMouse.merge(mouseParalogdNdS, left_on=["Gene stable ID", "Mouse paralogue gene stable ID"], right_on=["Gene 2", "Gene 1"], how="left").drop(columns=["Gene 1", "Gene 2"]).rename(columns={1: "dN/dS 1", 2:"dN 1", 3:"dS 1"})
masterTableMouse["dN/dS"] = masterTableMouse["dN/dS"].fillna(masterTableMouse["dN/dS 1"])
masterTableMouse["dN"] = masterTableMouse["dN"].fillna(masterTableMouse["dN 1"])
masterTableMouse["dS"] = masterTableMouse["dS"].fillna(masterTableMouse["dS 1"])
masterTableMouse.drop(columns=["dN/dS 1", "dN 1", "dS 1"], inplace=True)

masterTableHuman = masterTableHuman.merge(humanParalogProtID, left_on=["Gene stable ID", "Human paralogue gene stable ID"], right_on=["Gene 1", "Gene 2"], how="left").drop(columns=["Gene 1", "Gene 2"]).rename(columns={1: "% Protein Identity"})
masterTableHuman = masterTableHuman.merge(humanParalogProtID, left_on=["Gene stable ID", "Human paralogue gene stable ID"], right_on=["Gene 2", "Gene 1"], how="left").drop(columns=["Gene 1", "Gene 2"]).rename(columns={1: "% Protein Identity 1"})
masterTableHuman["% Protein Identity"] = masterTableHuman["% Protein Identity"].fillna(masterTableHuman["% Protein Identity 1"])
masterTableHuman.drop(columns=["% Protein Identity 1"], inplace=True)

masterTableMouse = masterTableMouse.merge(mouseParalogProtID, left_on=["Gene stable ID", "Mouse paralogue gene stable ID"], right_on=["Gene 1", "Gene 2"], how="left").drop(columns=["Gene 1", "Gene 2"]).rename(columns={1: "% Protein Identity"})
masterTableMouse = masterTableMouse.merge(mouseParalogProtID, left_on=["Gene stable ID", "Mouse paralogue gene stable ID"], right_on=["Gene 2", "Gene 1"], how="left").drop(columns=["Gene 1", "Gene 2"]).rename(columns={1: "% Protein Identity 1"})
masterTableMouse["% Protein Identity"] = masterTableMouse["% Protein Identity"].fillna(masterTableMouse["% Protein Identity 1"])
masterTableMouse.drop(columns=["% Protein Identity 1"], inplace=True)

display(masterTableHuman)
display(masterTableMouse)

,Gene stable ID,Human paralogue gene stable ID,Human paralogue homology type,Paralogue last common ancestor with Human,Gene 1 Brain,Gene 1 Colon,Gene 1 Esophagus,Gene 1 Heart,Gene 1 Kidney,Gene 1 Liver,...,PearDist,TEC,Number of Duplicates,Exon stable ID,Number of Exons,Paralogy Type,dN/dS,dN,dS,% Protein Identity
0,ENSG00000210049,NaN,NaN,NaN,17.516844,1.400129,0.999184,3.366039,4.574239,0.597851,...,NaN,NaN,0,ENSE00001544501,1,NA,NaN,NaN,NaN,NaN
1,ENSG00000211459,NaN,NaN,NaN,23502.966797,3544.858643,3435.218750,7646.939941,10045.823242,3707.004150,...,NaN,NaN,0,ENSE00001544499,1,NA,NaN,NaN,NaN,NaN
2,ENSG00000210077,NaN,NaN,NaN,18.865494,1.571674,1.380163,3.266439,3.701754,0.723936,...,NaN,NaN,0,ENSE00001544498,1,NA,NaN,NaN,NaN,NaN
3,ENSG00000210082,NaN,NaN,NaN,72589.617188,18863.207031,16807.601562,39093.738281,51649.222656,20325.992188,...,NaN,NaN,0,ENSE00001544497,1,NA,NaN,NaN,NaN,NaN
4,ENSG00000209082,NaN,NaN,NaN,25.209278,6.738225,4.899840,15.493462,11.392898,4.587103,...,NaN,NaN,0,ENSE00002006242,1,NA,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3625954,ENSG00000157873,ENSG00000243509,other_paralog,Chordata,12.655311,120.186638,69.148285,20.617067,101.015106,54.591118,...,0.463805,0.142857,21,"ENSE00001759361, ENSE00001576845, ENSE00003529...",79,out-paralog,0.05363,0.7226,13.4739,34.241245
3625955,ENSG00000157873,ENSG00000026103,other_paralog,Chordata,12.655311,120.186638,69.148285,20.617067,101.015106,54.591118,...,0.198451,0.214286,21,"ENSE00001759361, ENSE00001576845, ENSE00003529...",79,out-paralog,0.06168,0.7336,11.8944,35.319149
3625956,ENSG00000157873,ENSG00000120949,other_paralog,Chordata,12.655311,120.186638,69.148285,20.617067,101.015106,54.591118,...,0.937936,0.428571,21,"ENSE00001759361, ENSE00001576845, ENSE00003529...",79,out-paralog,0.00471,0.7275,154.5621,30.916031
3625957,ENSG00000132676,NaN,NaN,NaN,23.821829,46.711121,42.119331,28.719112,30.156294,29.079199,...,NaN,NaN,0,"ENSE00001936122, ENSE00001845528, ENSE00001811...",131,NA,NaN,NaN,NaN,NaN


,Gene stable ID,Mouse paralogue gene stable ID,Mouse paralogue homology type,Paralogue last common ancestor with Mouse,Gene 1 Brain,Gene 1 Colon,Gene 1 Esophagus,Gene 1 Heart,Gene 1 Kidney,Gene 1 Liver,...,PearDist,TEC,Number of Duplicates,Exon stable ID,Number of Exons,Paralogy Type,dN/dS,dN,dS,% Protein Identity
0,ENSMUSG00000064336,NaN,NaN,NaN,13.858894,1.109168,1.554887,1.335080,2.510688,1.941366,...,NaN,NaN,0,ENSMUSE00000521514,1,NA,NaN,NaN,NaN,NaN
1,ENSMUSG00000064337,NaN,NaN,NaN,6650.162769,1137.804863,980.724698,3528.113526,1768.333371,1315.123969,...,NaN,NaN,0,ENSMUSE00000521515,1,NA,NaN,NaN,NaN,NaN
2,ENSMUSG00000064338,NaN,NaN,NaN,8.313387,0.000000,0.689367,4.524155,2.143020,1.435777,...,NaN,NaN,0,ENSMUSE00000521516,1,NA,NaN,NaN,NaN,NaN
3,ENSMUSG00000064339,NaN,NaN,NaN,7030.316708,1823.723689,1022.478876,6443.836189,2117.191903,2070.449196,...,NaN,NaN,0,ENSMUSE00000521517,1,NA,NaN,NaN,NaN,NaN
4,ENSMUSG00000064340,NaN,NaN,NaN,420.723675,204.564568,245.541007,1503.662783,403.739623,145.173194,...,NaN,NaN,0,ENSMUSE00000521518,1,NA,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2426744,ENSMUSG00000026833,ENSMUSG00000022026,other_paralog,Bilateria,918.399401,16.312056,17.256652,4.852001,10.824372,1.124305,...,1.092443,0.250000,10,"ENSMUSE00001242819, ENSMUSE00001266606, ENSMUS...",14,out-paralog,0.00625,0.9111,145.7401,26.808511
2426745,ENSMUSG00000026833,ENSMUSG00000027848,other_paralog,Bilateria,918.399401,16.312056,17.256652,4.852001,10.824372,1.124305,...,0.714240,0.083333,10,"ENSMUSE00001242819, ENSMUSE00001266606, ENSMUS...",14,out-paralog,0.00419,0.6408,152.8687,35.195531
2426746,ENSMUSG00000026833,ENSMUSG00000038463,other_paralog,Bilateria,918.399401,16.312056,17.256652,4.852001,10.824372,1.124305,...,1.093196,0.083333,10,"ENSMUSE00001242819, ENSMUSE00001266606, ENSMUS...",14,out-paralog,0.00431,0.6725,155.9244,34.261242
2426747,ENSMUSG00000026833,ENSMUSG00000046167,other_paralog,Bilateria,918.399401,16.312056,17.256652,4.852001,10.824372,1.124305,...,0.525570,0.416667,10,"ENSMUSE00001242819, ENSMUSE00001266606, ENSMUS...",14,out-paralog,0.06974,0.8437,12.0974,27.886710


# Create Master Tables

In [217]:
newColOrderHuman = list(masterTableHuman.columns[0:4]) + list(masterTableHuman.columns[30:31]) + list(masterTableHuman.columns[22:28]) + list(masterTableHuman.columns[12:13]) + list(masterTableHuman.columns[21:22]) + list(masterTableHuman.columns[4:12]) + list(masterTableHuman.columns[13:21]) + list(masterTableHuman.columns[28:30]) + list(masterTableHuman.columns[31:35])
newColOrderMouse = list(masterTableMouse.columns[0:4]) + list(masterTableMouse.columns[30:31]) + list(masterTableMouse.columns[22:28]) + list(masterTableMouse.columns[12:13]) + list(masterTableMouse.columns[21:22]) + list(masterTableMouse.columns[4:12]) + list(masterTableMouse.columns[13:21]) + list(masterTableMouse.columns[28:30]) + list(masterTableMouse.columns[31:35])

masterTableHuman = masterTableHuman.loc[:, newColOrderHuman]
masterTableMouse = masterTableMouse.loc[:, newColOrderMouse]

In [218]:
masterTableHuman.to_csv("/Users/andrewhsu/Projects/McNair/data/humanParalogMasterTable.csv", index=False)
masterTableHuman.to_parquet("/Users/andrewhsu/Projects/McNair/data/humanParalogMasterTable.parquet", index=False)

masterTableMouse.to_csv("/Users/andrewhsu/Projects/McNair/data/mouseParalogMasterTable.csv", index=False)
masterTableMouse.to_parquet("/Users/andrewhsu/Projects/McNair/data/mouseParalogMasterTable.parquet", index=False)

In [219]:
display(masterTableHuman)
display(masterTableMouse)

,Gene stable ID,Human paralogue gene stable ID,Human paralogue homology type,Paralogue last common ancestor with Human,Paralogy Type,EuclidDist,EuclidDistNorm,EuclidDistLog,PearDist,TEC,...,Gene 2 Kidney,Gene 2 Liver,Gene 2 Pancreas,Gene 2 Stomach,Exon stable ID,Number of Exons,dN/dS,dN,dS,% Protein Identity
0,ENSG00000210049,NaN,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,ENSE00001544501,1,NaN,NaN,NaN,NaN
1,ENSG00000211459,NaN,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,ENSE00001544499,1,NaN,NaN,NaN,NaN
2,ENSG00000210077,NaN,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,ENSE00001544498,1,NaN,NaN,NaN,NaN
3,ENSG00000210082,NaN,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,ENSE00001544497,1,NaN,NaN,NaN,NaN
4,ENSG00000209082,NaN,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,ENSE00002006242,1,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3625954,ENSG00000157873,ENSG00000243509,other_paralog,Chordata,out-paralog,189.903580,0.500351,10.446088,0.463805,0.142857,...,4.668557,0.847602,1.235028,4.021246,"ENSE00001759361, ENSE00001576845, ENSE00003529...",79,0.05363,0.7226,13.4739,34.241245
3625955,ENSG00000157873,ENSG00000026103,other_paralog,Chordata,out-paralog,196.172974,0.323165,12.601530,0.198451,0.214286,...,1.590631,1.228026,0.426707,1.094653,"ENSE00001759361, ENSE00001576845, ENSE00003529...",79,0.06168,0.7336,11.8944,35.319149
3625956,ENSG00000157873,ENSG00000120949,other_paralog,Chordata,out-paralog,198.702774,0.713202,14.173954,0.937936,0.428571,...,0.412732,0.098749,0.078418,1.143263,"ENSE00001759361, ENSE00001576845, ENSE00003529...",79,0.00471,0.7275,154.5621,30.916031
3625957,ENSG00000132676,NaN,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,"ENSE00001936122, ENSE00001845528, ENSE00001811...",131,NaN,NaN,NaN,NaN


,Gene stable ID,Mouse paralogue gene stable ID,Mouse paralogue homology type,Paralogue last common ancestor with Mouse,Paralogy Type,EuclidDist,EuclidDistNorm,EuclidDistLog,PearDist,TEC,...,Gene 2 Kidney,Gene 2 Liver,Gene 2 Pancreas,Gene 2 Stomach,Exon stable ID,Number of Exons,dN/dS,dN,dS,% Protein Identity
0,ENSMUSG00000064336,NaN,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,ENSMUSE00000521514,1,NaN,NaN,NaN,NaN
1,ENSMUSG00000064337,NaN,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,ENSMUSE00000521515,1,NaN,NaN,NaN,NaN
2,ENSMUSG00000064338,NaN,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,ENSMUSE00000521516,1,NaN,NaN,NaN,NaN
3,ENSMUSG00000064339,NaN,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,ENSMUSE00000521517,1,NaN,NaN,NaN,NaN
4,ENSMUSG00000064340,NaN,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,ENSMUSE00000521518,1,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2426744,ENSMUSG00000026833,ENSMUSG00000022026,other_paralog,Bilateria,out-paralog,916.588582,1.313149,10.015711,1.092443,0.250000,...,15.643258,0.219291,0.411394,6.718520,"ENSMUSE00001242819, ENSMUSE00001266606, ENSMUS...",14,0.00625,0.9111,145.7401,26.808511
2426745,ENSMUSG00000026833,ENSMUSG00000027848,other_paralog,Bilateria,out-paralog,908.057141,1.040515,6.428900,0.714240,0.083333,...,13.253190,0.866962,0.266533,6.503741,"ENSMUSE00001242819, ENSMUSE00001266606, ENSMUS...",14,0.00419,0.6408,152.8687,35.195531
2426746,ENSMUSG00000026833,ENSMUSG00000038463,other_paralog,Bilateria,out-paralog,912.359159,1.274453,7.389735,1.093196,0.083333,...,1.881622,0.266059,0.678200,7.194103,"ENSMUSE00001242819, ENSMUSE00001266606, ENSMUS...",14,0.00431,0.6725,155.9244,34.261242
2426747,ENSMUSG00000026833,ENSMUSG00000046167,other_paralog,Bilateria,out-paralog,918.024404,0.927514,11.691460,0.525570,0.416667,...,0.006148,0.031491,0.000000,0.005854,"ENSMUSE00001242819, ENSMUSE00001266606, ENSMUS...",14,0.06974,0.8437,12.0974,27.886710


# Statistical Testing

In [18]:
statisticsDFHuman = pd.DataFrame({
    "Category": ["out-paralog", "in-paralog"],
    "n": 0.0,
    "Median": 0.0,
    "Mean": 0.0, 
    "Rho": 0.0,
    "Corr PValue" : 0.0
})

statisticsDFMouse = pd.DataFrame({
    "Category": ["out-paralog", "in-paralog"],
    "n": 0.0,
    "Median": 0.0,
    "Mean": 0.0, 
    "Rho": 0.0,
    "Corr PValue" : 0.0
})

In [19]:
sigHumanDF = pd.DataFrame({
    "Group 1": ["out-paralog"],
    "Group 2": ["in-paralog"],
    "PValue": 0.0
})

sigMouseDF = pd.DataFrame({
    "Group 1": ["out-paralog"],
    "Group 2": ["in-paralog"],
    "PValue": 0.0
})


In [20]:
dfListHuman = []
dfListMouse = []
for distMetric in masterTableHuman.columns[5:10]:
    nHumanArr = []
    nMouseArr = []
    medianHumanArr = []
    medianMouseArr = []
    meanHumanArr = []
    meanMouseArr = []
    corrHumanArr = []
    corrMouseArr = []
    pValueHumanArr = []
    pValueMouseArr = []

    for category in statisticsDFHuman["Category"]:
        filteredHumanDF = masterTableHuman[masterTableHuman["Paralogy Type"] == category][distMetric].dropna()
        filteredMouseDF = masterTableMouse[masterTableMouse["Paralogy Type"] == category][distMetric].dropna()

        nHumanArr.append(filteredHumanDF.shape[0])
        nMouseArr.append(filteredMouseDF.shape[0])
        medianHumanArr.append(filteredHumanDF.median())
        medianMouseArr.append(filteredMouseDF.median())
        meanHumanArr.append(filteredHumanDF.mean())
        meanMouseArr.append(filteredMouseDF.mean())

        corrHuman = stats.spearmanr(masterTableHuman[(masterTableHuman["Paralogy Type"] == category) & (~masterTableHuman[distMetric].isna())][distMetric], masterTableHuman[(masterTableHuman["Paralogy Type"] == category) & (~masterTableHuman[distMetric].isna())]["Number of Duplicates"])
        corrMouse = stats.spearmanr(masterTableMouse[(masterTableMouse["Paralogy Type"] == category) & (~masterTableMouse[distMetric].isna())][distMetric], masterTableMouse[(masterTableMouse["Paralogy Type"] == category) & (~masterTableMouse[distMetric].isna())]["Number of Duplicates"])
        corrHumanArr.append(corrHuman[0])
        corrMouseArr.append(corrMouse[0])
        pValueHumanArr.append(corrHuman[1])
        pValueMouseArr.append(corrMouse[1])

    statisticsDFHuman["n"] = nHumanArr
    statisticsDFHuman["Median"] = medianHumanArr
    statisticsDFHuman["Mean"] = meanHumanArr
    statisticsDFHuman["Rho"] = corrHumanArr
    statisticsDFHuman["Corr PValue"] = pValueHumanArr
    statisticsDFMouse["n"] = nMouseArr
    statisticsDFMouse["Median"] = medianMouseArr
    statisticsDFMouse["Mean"] = meanMouseArr
    statisticsDFMouse["Rho"] = corrMouseArr
    statisticsDFMouse["Corr PValue"] = pValueMouseArr

    dfListHuman.append(statisticsDFHuman.copy())
    dfListMouse.append(statisticsDFMouse.copy())

In [21]:
dfListHuman[0]

,Category,n,Median,Mean,Rho,Corr PValue
0,out-paralog,3228272,0.810108,0.810108,0.178815,0.000000e+00
1,in-paralog,6890,0.431129,0.518971,-0.065326,5.721087e-08


In [22]:
dfListHumanSig = []
dfListMouseSig = []
for distMetric in masterTableHuman.columns[5:10]:
    pValueHumanArr = []
    pValueMouseArr = []
    for idx in range(len(sigHumanDF["Group 1"])):
        filteredHumanDF1 = masterTableHuman[masterTableHuman["Paralogy Type"] == sigHumanDF["Group 1"][idx]][distMetric].dropna()
        filteredHumanDF2 = masterTableHuman[masterTableHuman["Paralogy Type"] == sigHumanDF["Group 2"][idx]][distMetric].dropna()
        filteredMouseDF1 = masterTableMouse[masterTableMouse["Paralogy Type"] == sigMouseDF["Group 1"][idx]][distMetric].dropna()
        filteredMouseDF2 = masterTableMouse[masterTableMouse["Paralogy Type"] == sigMouseDF["Group 2"][idx]][distMetric].dropna()

        humanSig = stats.mannwhitneyu(filteredHumanDF1, filteredHumanDF2)[1]
        mouseSig = stats.mannwhitneyu(filteredMouseDF1, filteredMouseDF2)[1]

        pValueHumanArr.append(humanSig)
        pValueMouseArr.append(mouseSig)
    sigHumanDF["PValue"] = pValueHumanArr
    sigMouseDF["PValue"] = pValueMouseArr

    dfListHumanSig.append(sigHumanDF.copy())
    dfListMouseSig.append(sigMouseDF.copy())
        

In [23]:
with pd.ExcelWriter("/Users/andrewhsu/Projects/McNair/data/humanParalogStatistics.xlsx") as w:
    for idx, distMetric in enumerate(masterTableHuman.columns[5:10]):
        dfListHuman[idx].to_excel(w, sheet_name=distMetric, index=False)
        dfListHumanSig[idx].to_excel(w, sheet_name=distMetric, index=False, startrow=0, startcol=7)
    
with pd.ExcelWriter("/Users/andrewhsu/Projects/McNair/data/mouseParalogStatistics.xlsx") as w:
    for idx, distMetric in enumerate(masterTableMouse.columns[5:10]):
        dfListMouse[idx].to_excel(w, sheet_name=distMetric, index=False)
        dfListMouseSig[idx].to_excel(w, sheet_name=distMetric, index=False, startrow=0, startcol=7)